# Venusian PCW Detector · 数据与模型工作流

原始 MAG 由使用者提供。缓存、特征、模型分数和搜索结果在本地重建。默认只校验权重与事件清单；在参数单元开启所需步骤。绘图见 [figures.ipynb](figures.ipynb)。

In [ ]:
# 数据和运行环境
MAG_DIR = None  # 填 1 Hz VSO BAM 目录。None 沿用 VENUS_EXPRESS_RAW_DATA_DIR。
EVENTS_CSV = None  # 只影响缓存、目录和波谱单元。None 使用随仓库提供的事件清单。
DEVICE = "cpu"  # 模型训练、评分和推理用的设备，可选 cpu、mps、cuda。
WORKERS = 2  # 数据处理的并行进程数。

# 【缓存计算】从原始 MAG 生成可复用的日特征，输出到 build/feature_cache/。
RUN_CACHE = False
RUN_CATALOGUE = False  # 从事件时段重算波能、位置和密度。
RUN_PSD_AUDIT = False  # 对事件时段重算并检查全频 PSD。
EVENT_DAYS = ["2006-04-24"]  # 缓存、目录和波谱共用这些日期。None 使用清单中的全部日期。

# 【模型训练】首次先重建训练和验证窗口，再开启训练。
STAGE = "MStandard"  # MSeed、MExtend1、MExtend2、MStandard 或 holdout2013。
TRAIN_SEEDS = [2026]  # 每个种子单独训练。holdout2013 使用 2026。
PREPARE_TRAINING_WINDOWS = False  # 原始 BAM + 窗口清单 → build/window_features/。
RUN_TRAIN = False  # 读取重建窗口，训练 100 轮，保存权重和验证阈值。

# 【固定窗口评价】首次先构建验证集和固定测试集，再运行评分。
RUN_EVALUATION = False  # 用仓库提供的四阶段权重评分。
PREPARE_EVALUATION_WINDOWS = False  # 从 BAM 重建四阶段的验证集及固定测试集。
EVAL_TRAINED_SEED = None  # 如填 2026，则额外评价 STAGE 对应的本地训练权重。

# 【连续推理】从 BAM 滑窗评分，再细化候选边界、执行物理核验。
RUN_SEARCH = False
SEARCH_SCOPE = "mission"  # mission 使用任务期权重，holdout 使用 2013 留出权重。
SEARCH_DAYS = ["2007-02-04"]  # 独立于 EVENT_DAYS。holdout 可选 ["2013-11-21"]，None 扫全范围。
RESUME_SEARCH = False  # 续跑时沿用同一数据源、日期和设备。
RUN_EXPOSURE = False  # 从 BAM 单独计算覆盖时间和空间曝光。
RUN_OCCURRENCE = False  # 在已有 mission 搜索结果后运行，按该次搜索日期汇总曝光和发生率。

## 1. 校验输入

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

from venus_express.repro.settings import BUILD, display_path, read, write
from venusian_pcw.notebook import day_flags, run_cli

# 把 Notebook 选定的 BAM 目录传给随后启动的命令。
if MAG_DIR is not None:
    os.environ["VENUS_EXPRESS_RAW_DATA_DIR"] = str(Path(MAG_DIR).expanduser().resolve())

os.environ.setdefault("MPLCONFIGDIR", str(BUILD / "matplotlib"))
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

# 校验附带的权重和事件清单。实际计算由参数开关控制。
run_cli("verify")

## 2. 缓存、事件目录和波谱

这些步骤读取原始 MAG。设置 `EVENT_DAYS` 可先处理少量观测日；改为 `None` 则按事件清单处理全部日期。逐事件结果保存在 `build/raw_catalogue/`，单日或多日诊断写入其 `diagnostic/` 子目录。

In [ ]:
# 自带清单和用户清单共用同一个 CSV 接口。
event_flags = ["--events", Path(EVENTS_CSV).expanduser().resolve()] if EVENTS_CSV else []
days = day_flags(EVENT_DAYS)
# 按 UTC 日生成磁场和偏振特征，后续步骤按需复用。
if RUN_CACHE:
    run_cli("cache", *event_flags, *days, "--workers", WORKERS)
# 按事件清单测量原始数据，再读取本次生成的统计摘要。
if RUN_CATALOGUE:
    run_cli("catalogue", *event_flags, *days, "--workers", WORKERS)
    output = BUILD / "raw_catalogue"
    if EVENT_DAYS:
        output /= "diagnostic"
    display(pd.json_normalize(read(output / "summary.json")))
# 从事件波形复核全频谱峰与回旋频带。
if RUN_PSD_AUDIT:
    run_cli("psd-audit", *event_flags, *days, "--workers", WORKERS)
    display(pd.json_normalize(read(BUILD / "raw_psd_summary.json")))

## 3. 构建窗口与训练

从 BAM 和窗口清单重建训练及验证特征。首次运行时先打开 `PREPARE_TRAINING_WINDOWS`，再开启 `RUN_TRAIN`。本地训练输出包含验证记录、选中的检查点及验证阈值。每个任务运行 100 个 epoch。`holdout2013` 使用种子 2026。

In [ ]:
training_root = BUILD / "training_from_bam"

def training_run(seed):
    return training_root / STAGE / f"seed{seed}"

# 按所选阶段的窗口清单提取训练和验证特征。
if PREPARE_TRAINING_WINDOWS:
    run_cli("prepare-windows", STAGE)
# 按种子独立训练。每次运行会写入自己的阶段和种子目录。
if RUN_TRAIN:
    for seed in TRAIN_SEEDS:
        run_cli("train", STAGE, "--seed", seed, "--device", DEVICE)
        run = training_run(seed)
        print(f"检查点 {display_path(run / 'selected.pt')}")
        # 查看最后几轮的验证记录，以及所选检查点的验证阈值。
        display(pd.read_json(run / "history.json").tail())
        display(pd.DataFrame(read(run / "selection.json")["validation_cuts"]))

## 4. 固定窗口评价

开启 `PREPARE_EVALUATION_WINDOWS` 可重建四阶段的验证窗口及固定测试窗口，再用 `RUN_EVALUATION` 计算参考权重的评价表。设置 `EVAL_TRAINED_SEED` 可另外评价某个本地训练检查点。2013 留出模型使用单独的连续记录协议。

In [ ]:
# 为参考权重重建验证窗口，固定测试窗口由各阶段共用。
if PREPARE_EVALUATION_WINDOWS:
    for stage in ("MSeed", "MExtend1", "MExtend2", "MStandard"):
        run_cli("prepare-windows", stage, "--role", "validation")
    run_cli("prepare-windows", "fixed_test")

# 输出验证工作点指标和固定测试集上的等召回率比较。
if RUN_EVALUATION:
    run_cli("evaluate-windows", "--device", DEVICE)
    evaluation = read(BUILD / "windows_from_bam" / "summary.json")
    display(pd.DataFrame(evaluation["validation_operating_point"]))
    display(pd.DataFrame(evaluation["equal_test_recall"]))

# 使用本地训练目录下的 selected.pt 独立进行固定窗口评分。
if EVAL_TRAINED_SEED is not None:
    if STAGE == "holdout2013":
        raise ValueError("请用 2013 留出连续记录协议评价 holdout2013")
    import torch
    from venus_express.repro.model import build_model
    from venus_express.repro.training import arrays, predict
    from venusian_pcw.window_inputs import prepared_path
    from venusian_pcw.windows import measures

    run = training_run(EVAL_TRAINED_SEED)
    selection = read(run / "selection.json")
    # 从该次训练的验证记录读取 80% 召回率对应阈值，再到固定测试集评价。
    cutoff = next(row["logit"] for row in selection["validation_cuts"] if row["recall"] == 0.8)
    test = arrays(prepared_path(None, None))
    model = build_model()
    model.load_state_dict(torch.load(run / "selected.pt", map_location="cpu", weights_only=True))
    scores = pd.DataFrame(dict(source=test["source"], logit=predict(model.to(DEVICE), test, DEVICE)))
    result = dict(stage=STAGE, seed=EVAL_TRAINED_SEED, threshold_logit=cutoff, **measures(scores, cutoff))
    write(run / "fixed_test_metrics.json", result)
    display(pd.DataFrame([result]))

## 5. 连续搜索与空间曝光

`SEARCH_DAYS = None` 使用原始数据目录内该范围的全部观测日。任务期推理直接使用随附权重。留出搜索前先用 `prepare-windows holdout2013 --role validation` 重建验证特征，搜索阈值由该验证集重新计算。开启 `RUN_OCCURRENCE` 可汇总本地任务期搜索并计算与该次搜索日期匹配的曝光。

In [ ]:
# 使用所选范围的随附权重。候选、边界和物理核验结果写入 build/raw/。
if RUN_SEARCH:
    flags = ["--resume"] if RESUME_SEARCH else []
    run_cli("search", SEARCH_SCOPE, *day_flags(SEARCH_DAYS), "--device", DEVICE, "--workers", WORKERS, *flags)
    complete = read(BUILD / "raw" / SEARCH_SCOPE / "complete.json")
    print(f"已搜索 {len(complete['inputs']['days'])} 天，得到 {complete['parents']} 个父候选。")
# 单独扫描观测记录，计算可用的逐日和空间覆盖。
if RUN_EXPOSURE:
    run_cli("exposure", "--workers", WORKERS)
    print(display_path(BUILD / "raw" / "exposure"))
# 读取已完成的任务期搜索结果，结合对应日期的曝光统计发生率。
if RUN_OCCURRENCE:
    run_cli("replay", "--workers", WORKERS)
    display(pd.json_normalize(read(BUILD / "summary.json")))